In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
SILVER_PATH_CLASSE    = "workspace.case_spark_cvm.silver_registro_classe_cvm"
SILVER_PATH_FUNDO     = "workspace.case_spark_cvm.silver_registro_fundo_cvm"
SILVER_PATH_SUBCLASSE = "workspace.case_spark_cvm.silver_registro_subclasse_cvm"

NOME_TABELA  = f"gold_dim_fundo" 
GOLD_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

Tabela dimensão mestre. Une as 4 tabelas de cadastro e resolve o join uma única vez. 

In [0]:
df_classe_silver    = spark.read.table(SILVER_PATH_CLASSE)
df_fundo_silver     = spark.read.table(SILVER_PATH_FUNDO)
df_subclasse_silver = spark.read.table(SILVER_PATH_SUBCLASSE)

In [0]:
df_classe_silver.filter(f.col("cnpj_classe") == "41240381000163").display()

In [0]:
df_subclasse_silver.filter(f.col("id_registro_classe") == "13581").display()

### Joins

In [0]:
# A Classe é a base absoluta (onde vive o CNPJ que vai pro Power BI)
df_dim_fundo = df_classe_silver.alias("c") \
    .join(
        df_fundo_silver.alias("fu"), 
        f.col("c.id_registro_fundo") == f.col("fu.id_registro_fundo"), 
        "left"
    ) \
    .join(
        df_subclasse_silver.alias("sc"), 
        f.col("c.id_registro_classe") == f.col("sc.id_registro_classe"), 
        "left"
    )

df_dim_fundo = df_dim_fundo.select(
    # Chaves de Relacionamento
    f.col("id_subclasse"),
    f.col("c.id_registro_classe"),
    f.col("fu.id_registro_fundo"),
    f.col("c.cnpj_classe").alias("cnpj_fundo_classe"), 
    # Se a subclasse não existir, pega a denominação do fundo para não ficar em branco no BI
    f.coalesce(
        f.col("sc.denominacao_social"), 
        f.col("fu.denominacao_social")
    ).alias("denominacao_social"),

    f.col("c.situacao").alias("situacao"),
    # Características de Negócio (Nível Classe/Subclasse)
    f.col("sc.data_inicio"),
    f.col("fu.tipo_fundo"),
    f.col("c.tipo_classe"),
    f.col("c.classe_cotas"),
    f.col("c.classificacao"),
    f.col("c.classificacao_anbima"),
    f.col("c.forma_condominio"),
    f.col("c.classe_esg"),
    f.col("c.exclusivo"),
    f.col("c.publico_alvo"),
    f.col("c.custodiante"),
    # Governança (Nível Fundo)
    f.col("fu.gestor"),
    f.col("fu.administrador"),
    f.col("c.indicador_desempenho")
)

# Se um CNPJ tiver mais de uma linha, organiza pela data_inicio mais recente
window_dedup = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("data_inicio").desc_nulls_last())

df_dim_fundo = df_dim_fundo \
    .withColumn("rn", f.row_number().over(window_dedup)) \
    .filter(f.col("rn") == 1) \
    .drop("rn")


### 6. Criação da coluna benchmark_disponivel

In [0]:
df_dim_fundo = df_dim_fundo.withColumn(
    "benchmark_normalizado",
    # 1. CDI e Derivados
    f.when(f.col("indicador_desempenho").isin("DI de um dia", "Taxa Básica Financeira", "Taxa Anbid"), "CDI")\
    
    # 2. IPCA e Derivados (NTN-B entra aqui ANTES da regra do Andima)
    .when(f.col("indicador_desempenho").contains("NTN-B"), "IPCA")\
    .when(
        f.col("indicador_desempenho").isin(
            "Índice de Preços ao Consumidor Amplo (IPCA/IBGE)",
            "Índice Nacional de Preços ao Consumidor (INPC/IBGE)",
            "Índice de preços"
        ), "IPCA"
    )\
    
    # 3. Regras de "contains" genéricas (agora seguras porque tiramos a NTN-B)
    .when(f.col("indicador_desempenho").contains("Andima"), "CDI")\
    .when(f.col("indicador_desempenho").contains("Anbid"), "CDI")\
    
    # 4. Selic e Ibovespa
    .when(f.col("indicador_desempenho").isin("Taxa Selic"), "Selic")\
    .when(f.col("indicador_desempenho").isin("Ibovespa", "IBrX", "IBrX-50"), "Ibovespa")\
    
    # 5. Outros Índices Específicos que vieram no CSV
    .when(f.col("indicador_desempenho").isin("Índice Geral de Preços-Mercado (IGP-M)", "Índice de Mercado Andima NTN-C até 5 anos"), "IGP-M")\
    .when(f.col("indicador_desempenho") == "Taxa Referencial", "TR")\
    .when(f.col("indicador_desempenho") == "Dólar comercial", "Dolar")\
    
    # 6. Sem Benchmark Declarado
    .when(
        f.col("indicador_desempenho").isin("Não se aplica", "OUTROS") | 
        f.col("indicador_desempenho").isNull(), 
        "SEM_BENCHMARK"
    )\
    .otherwise("NAO_DISPONIVEL")
)\
.withColumn(
    "benchmark_disponivel",
    # Correção do Bug Case-Sensitive (Claude O3): Deve bater EXATAMENTE com o que geramos acima
    f.when(
        f.col("benchmark_normalizado").isin("CDI", "Selic", "IPCA", "Ibovespa"),
        "S"
    ).otherwise("N")
)

In [0]:
df_dim_fundo.display()

In [0]:

log.info(f"Iniciando a escrita da dimensão unificada em: {GOLD_PATH}")

PipelineConfig.gravar_dimensao_gold(
    spark=spark,
    df_novo=df_dim_fundo,
    tabela_destino=GOLD_PATH,
    chave_pk=['cnpj_fundo_classe']

)

log.info(f"Processamento da {GOLD_PATH} concluído com sucesso!")

In [0]:
%sql
select
    *
from workspace.case_spark_cvm.silver_quarentena

